# IAD Pipeline — Training
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

This notebook uses the refactored `AnomalyDetectionManager`:
- `train()` / `eval()` no longer take a `tiling` flag or call `adjustPaths()` manually — tiled-ensemble is currently the only supported mode, and paths are resolved internally by `_prepareRun`.
- Before calling an action, we check `manager.can_run(...)` / `manager.get_missing_requirements(...)` so we can show the user *why* something isn't ready yet instead of hitting a bare exception.
- State-related failures raise `ManagerStateError` subclasses (`NoModelLoadedError`, `NoDatasetLoadedError`, `TilingNotConfiguredError`, `ModelNotTrainedError`, `CheckpointNotFoundError`), each carrying a `.missing` list.

## 1. Environment Setup
Configure database URI and API keys.

In [ ]:
import os
import sys
import warnings



# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'

sys.path.append("src")

import wandb
import logging
from pathlib import Path

from src.manager import AnomalyDetectionManager as ADM
from src.manager import DatasetSession as DS
from src.manager import (
    ManagerState,
    ManagerError,
    ManagerStateError,
    ConfigError,
    NoModelLoadedError,
    NoDatasetLoadedError,
    TilingNotConfiguredError,
    ModelNotTrainedError,
    CheckpointNotFoundError,
)
from src.tiling.tilingCheckpoints import checkTiledCheckpointsExist


warnings.filterwarnings("ignore", category=FutureWarning, module="timm.models.layers")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="openvino.runtime")

logger = logging.getLogger("logger")

wandb.login()

## 2. Configuration
Set your run parameters here before executing the pipeline.

`ADM.loadProduct(...)` loads the model, configures tiling, and resolves output/checkpoint
paths for you — no manual `adjustPaths()` call needed afterwards.

In [ ]:
from src.userConfigs import Product

datasetDir  = Path("datasets/")
configDir   = Path("configs/")
outputPath  = Path("results/")
productPath = Path("Products/fabric_03.yaml")
productConfigPath = Path(configDir / productPath)

product: Product
manager, product = ADM.loadProduct(
    productConfigPath=productConfigPath,
    outputPath=outputPath,
    configDir=configDir,
)

print(f"Loaded product: {product.name}")
print(product)
print(f"Manager state: {manager.state!r}")

## 3. Load & Select Dataset

We load the dataset session and select the product's category. `DatasetSession` is
independent of the manager's readiness state — you can load and inspect data before
a model is ready, or vice versa.

In [ ]:
datasetSession = DS.loadDatasetFromConfig(product.datasetConfig, overwrite=True, merge=False)
datasetSession.select_category(product.name)

print(f"Dataset: {datasetSession.datasetName}, category: {datasetSession.category}")
print(f"Images in view: {len(datasetSession.FO_Dataset)}")

## 3.1 Inspect Dataset

In [ ]:
datasetSession.launchSession()

## 4. Readiness Check

Before training, ask the manager what (if anything) is missing, rather than firing the
call and translating an exception. This is the check a UI would run to enable/disable
a "Train" button.

In [ ]:
manager.attachDatasetSession(datasetSession)
missing = manager.get_missing_requirements("train")
if missing:
    print("Not ready to train yet. Missing:")
    for item in missing:
        print(f"  - {item}")
else:
    print("Ready to train.")
# manager._prepareRun(trainerConfig=product.trainerConfig,
#     modelConfig=product.modelConfig,
#     datamoduleConfig=product.datamoduleConfig,
#     datasetSession=datasetSession,
#     tilingPipelineConfig=product.tilingPipelineConfig,)

## 5. Training

`train()` is tiled-ensemble only for now (the `tiling` parameter has been removed —
non-tiled support will be added later as an explicit branch, once implemented). Paths,
tiling setup, callbacks, and the W&B logger are all handled internally by `_prepareRun`.

We wrap the call so that a `ManagerStateError` (e.g. someone re-running this cell before
`generateModel`/`setupTiling` ran) produces a clear message instead of a raw traceback.

In [ ]:
try:

    manager.train(
        trainerConfig=product.trainerConfig,
        modelConfig=product.modelConfig,
        datamoduleConfig=product.datamoduleConfig,
        datasetSession=datasetSession,
        tilingPipelineConfig=product.tilingPipelineConfig,
    )
    print("Training complete.")
except ManagerStateError as e:
    print(f"Could not train: {e}")
    print(f"Missing: {e.missing}")
except ManagerError as e:
    print(f"Training failed: {e}")

print(f"Manager state: {manager.state!r}")

INFO: Dataset used for training: Name:        AITEX_-fabric_03
Media type:  image
Num samples: 51
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    anomalyType:      fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    category:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    split:            fiftyone.core.fields.StringField
    label_index:      fiftyone.core.fields.IntField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Segmentation)
    mask_path:        fif

Trainer: 0it [00:00, ?it/s]

INFO: Tiled ensemble training started. Separate models will be trained for 11 tile locations.
INFO: Initializing InpFormer model.


c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: Loaded model: dinov2reg_vit_base_14


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.
c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\lightning\pytorch\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 3. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
`Trainer.fit` stopped: `max_steps=1` reached.
Batch size 2 succeeded, trying batch size 4
`Trainer

In [ ]:
datasetSession.launchSession()
# datasetSession.save()

## 6. Evaluation

`eval()` requires the model to actually be trained (`ManagerState.TRAINED`), which is
only set once `train()` confirms a checkpoint landed on disk — not just that the call
returned. Check readiness first, same pattern as training.

In [ ]:
# ckptDir = Path("results/MVTecADShort/cable/Padim/tiled/checkpoints")
# ckptDir = manager.ckptDir
# if ckptDir is None:
#     raise ValueError
print(manager.modelTrainingDir)
print(manager.ckptDir)

if manager.modelTrainingDir is None and manager.ckptDir is not None:
    manager.modelTrainingDir = manager.ckptDir.parent
missing = manager.get_missing_requirements("eval")
print(missing)
# manager.loadCheckpoint(ckptDir, product.tilingPipelineConfig)

# if missing:
#     print("Not ready to evaluate yet. Missing:")
#     for item in missing:
#         print(f"  - {item}")

# if ManagerState.CHECKPOINT_AVAILABLE:
try:
    manager.eval(
        evalConfig=product.trainerConfig,
        modelConfig=product.modelConfig,
        datamoduleConfig=product.datamoduleConfig,
        datasetSession=datasetSession,
        tilingPipelineConfig=product.tilingPipelineConfig,
    )
    print("Evaluation complete.")
except ManagerStateError as e:
    print(f"Could not evaluate: {e}")
except ManagerError as e:
    print(f"Evaluation failed: {e}")

In [ ]:
datasetSession.launchSession()

## 7. Evaluate on Unknown / Prediction Data

Inference reads its checkpoint from an explicit `trainingDir` (which may belong to a
different manager/session than the one currently in memory) rather than relying on
`self.ckptDir` from the last training run. `inference()` checks that the checkpoint
file actually exists at that path and raises `CheckpointNotFoundError` if not — state
flags alone can't guarantee this, since `trainingDir` is caller-supplied.

This cell is left commented out, matching the placeholder in the original notebook — 
uncomment and adjust `predDatasetName` / `predDatasetDir` to run a prediction pass.

In [ ]:
predDatasetName = "MVTecADShortPred"
predDatasetDir = datasetDir / predDatasetName

predSession = DS.loadDatasetFromDisk(
    predDatasetDir,
    datasetName=predDatasetName,
    overwrite=True,
    merge=False,
    split=("pred",),
)
predSession.select_category(product.name)

manager.modelTrainingDir = product.refresh_training_dir(manager.baseOutputDir)

missing = manager.get_missing_requirements("inference")
if missing:
    print("Not ready for inference yet. Missing:")
    for item in missing:
        print(f"  - {item}")
else:
    try:
        manager.inference(
            inferencerConfig=product.inferencerConfig,
            modelConfig=product.modelConfig,
            modelTrainingDir=product.modelTrainingDir,
            datamoduleConfig=product.datamoduleConfig,
            datasetSession=predSession,
            tilingPipelineConfig=product.tilingPipelineConfig,
        )
        print("Inference complete.")
    except CheckpointNotFoundError as e:
        print(f"No checkpoint available: {e}")
    except ManagerStateError as e:
        print(f"Could not run inference: {e}")
    except ManagerError as e:
        print(f"Inference failed: {e}")

predSession.launchSession()